[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/77_dijkstra_free_edge_solution.ipynb)

# Solution: Shortest Path with One Free Edge

Reference solution — state-augmented (layered) Dijkstra.

## 解析

**结论：把「是否已用掉免费边」编进状态，做一次分层 Dijkstra。**

### 状态设计
普通最短路的状态是「当前在哪个点」。这里多了一个决策——那条可以置 0 的边用了没有——于是把状态扩成二元组 `(node, used)`：
- `used = 0`：还没用过免费边；
- `used = 1`：已经把某条边置 0 了。

相当于把原图复制成两层：**第 0 层**（没用免费边）和**第 1 层**（用过了）。总共 `2N` 个状态。

### 转移（对每条边 `(u, v, w)`）
- 从 `(u, 0)`：
  - **正常付费**走到 `(v, 0)`，代价 `w`（留在第 0 层）；
  - **在这条边上用掉免费边**，走到 `(v, 1)`，代价 `0`（跨层，把这条边当免费）。
- 从 `(u, 1)`：免费边已用完，只能
  - **正常付费**走到 `(v, 1)`，代价 `w`（留在第 1 层）。

只有 `0→1` 的跨层转移，没有 `1→0`，保证免费边最多用一次。

### 求解与答案
从 `(1, 0)` 出发跑一次 Dijkstra（**权重全为正**，Dijkstra 成立）。终点答案是 `min(dist[N][0], dist[N][1])`——分别对应「一条免费边都没用」和「用了一条」，两者都不可达则返回 `-1`。

> 为什么不用「枚举每条边置 0，各跑一次 Dijkstra」？那是 `O(M · (M+N)logN)`，边多时很慢。分层法把它压成一次 Dijkstra。（本题随机对拍就是用暴力枚举验证分层解的正确性。）

### 贪心直觉
免费边一定花在**所选路径上权重最大的那条边**才最省。但「哪条路径最优」本身又依赖免费边花在哪——两者耦合，所以不能先定路径再定免费边。分层 Dijkstra 用状态把这个耦合决策交给最短路自己去搜，天然得到全局最优。

### 边界
- `N == 1`：起点即终点，空路径，答案 `0`。
- 自环 / 平行边 / 环：Dijkstra 靠 `dist` 松弛与出堆判重天然处理，无需特判。

### 复杂度
状态数 `2N`、转移数 `2M`，Dijkstra 总复杂度 `O((N + M) log N)`，空间 `O(N + M)`。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import heapq
from typing import List, Tuple

In [ ]:
# ✅ SOLUTION

class Solution:
    def min_weight_path(self, n: int, edges: List[Tuple[int, int, int]]) -> int:
        adj = [[] for _ in range(n + 1)]
        for u, v, w in edges:
            adj[u].append((v, w))

        INF = float('inf')
        dist = [[INF, INF] for _ in range(n + 1)]   # dist[node][used]
        dist[1][0] = 0
        pq = [(0, 1, 0)]                            # (cost, node, used)

        while pq:
            d, u, used = heapq.heappop(pq)
            if d > dist[u][used]:
                continue                           # stale heap entry
            for v, w in adj[u]:
                # option 1: pay w, stay in the same layer
                if d + w < dist[v][used]:
                    dist[v][used] = d + w
                    heapq.heappush(pq, (d + w, v, used))
                # option 2: spend the free edge here (0 -> 1), cost 0
                if used == 0 and d < dist[v][1]:
                    dist[v][1] = d
                    heapq.heappush(pq, (d, v, 1))

        ans = min(dist[n][0], dist[n][1])
        return -1 if ans == INF else ans

In [ ]:
# Demo
sol = Solution()
print(sol.min_weight_path(3, [(1, 2, 5), (2, 3, 3)]))               # 3
print(sol.min_weight_path(4, [(1, 2, 1), (2, 3, 100), (3, 4, 1)]))  # 2
print(sol.min_weight_path(3, [(1, 2, 4)]))                          # -1

In [ ]:
from torch_judge import check
check('dijkstra_free_edge')